# НИС «Основы анализа данных в Python»

*Алла Тамбовцева*

## Практикум 6. Датафреймы `pandas`: формат дата-время и объединение датафреймов

В файле `hse.xlsx` (скачать [здесь](https://github.com/allatambov/PyDat25/blob/main/hse.xlsx)) сохранены посты сообщества [Цитатник ВШЭ](https://vk.com/hseteachers) ВКонтакте. На листе `posts` собрана основная информация по самим постам:

* `id`: id поста;
* `text`: текст поста;
* `nreposts`: число репостов;
* `nlikes`: число лайков;
* `nviews`: число просмотров.
* `tag`: первый тэг после текста.

На листе `dates` собрана информация по времени публикации постов в разных форматах:

* `id`: id поста;
* `timestamp`: метка времени в формате UNIX-времени (POSIX-времени);
* `date1`: дата публикации поста в формате ГГГГ-ММ-ДД;
* `date2`: дата публикации поста в формате ДД/ММ/ГГГГ.

Загрузите данные и поработайте с ними.

> Сохраните данные с первого листа файла Excel в датафрейм `df1`, а данные со второго листа – в датафрейм `df2`. Подсказка: функция `read_excel()` и аргумент `sheet_name`.

In [1]:
import pandas as pd

In [2]:
# в sheet_name индекс листа, можем вписать название,
# тогда будет sheet_name = 'posts'

df1 = pd.read_excel("hse.xlsx", sheet_name = 0)
df1.head()

,id,text,nreposts,nlikes,nviews,tag
0,40491,"Если вы общаетесь с Богом - это хорошо,А вот е...",8,20,1003.0,ВШЭСПБ
1,40489,Студенты: на лекции про революцию Мэйдзи в Япо...,2,30,1888.0,Суздальцев_ВШЭ
2,40488,«С точки зрения Сергея Караганова: ядерный уда...,10,24,1998.0,Суслов_ВШЭ
3,40485,"Вообще, расцвет математики и матанализа пришел...",11,32,2585.0,Лебедев_ВШЭ
4,40482,"Закон, как телефонный столб. Его невозможно пе...",10,40,3031.0,Гражданское_право


In [3]:
# в sheet_name индекс листа, можем вписать название,
# тогда будет sheet_name = 'dates'

df2 = pd.read_excel("hse.xlsx", sheet_name = 1)
df2.head()

,timestamp,date1,date2,id
0,1744982557,2025-04-18,18/04/2025,40491
1,1744900727,2025-04-17,17/04/2025,40489
2,1744460674,2025-04-12,12/04/2025,40488
3,1743855096,2025-04-05,05/04/2025,40485
4,1743420895,2025-03-31,31/03/2025,40482


### Часть 1: формат дата-время

Для начала поработаем с датафреймом `df2` с данными о времени публикации постов.

> Запросите типы столбцов `df2`.

In [4]:
# пока все даты записаны текстом (object)

print(df2.dtypes)

timestamp     int64
date1        object
date2        object
id            int64
dtype: object


Рассмотрим два разных варианта преобразования текста с датой внутри в полноценный формат *дата-время* (*datetime*).

In [5]:
# вариант 1 (менее универсальный)

df2["date_norm"] = df2["date1"].astype("datetime64[ns]")

> **Пояснения.** Просто меняем тип столбца с текстового на `datetime64`.

In [ ]:
# вариант 2 (более универсальный)

df2["date_norm"] = pd.to_datetime(df2["date2"], format = "%d/%m/%Y")

Про форматирование дат и принятые сокращения можно почитать [здесь](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior).

> **Пояснения.** Здесь символ `%` означает, что впереди идет не обычная буква, а специальное сокращение, принятое при форматировании дат. Так, `%d` означает, что на этом месте в строках внутри `date2` стоит число (номер дня), `%m` – месяц (номер месяца), `%Y` – год (в виде 4-значного числа). Все части даты разделены `/`, поэтому этот символ тоже оставляем в текстовом шаблоне `format` для описания строк с датами.

Теперь посмотрим на столбец `timestamp`.

> Выведите первые пять значений столбца. Что такое UNIX-время?

In [6]:
df2["timestamp"].head()

0    1744982557
1    1744900727
2    1744460674
3    1743855096
4    1743420895
Name: timestamp, dtype: int64

> **Ответ.** Время в формате UNIX– число секунд с 1 января 1970 года.

Переделаем целочисленную метку времени в более привычный формат фиксирования даты и времени:

In [7]:
# раз число секунд с 1 января 1970 года, вместо format для текстовых строк с датой
# используем аргумент unit и сообщаем, что единицы измерения в timestamp - секунды

df2["datetime"] = pd.to_datetime(df2['timestamp'], unit='s')

In [8]:
# теперь с типами все хорошо

print(df2.dtypes)

timestamp             int64
date1                object
date2                object
id                    int64
date_norm    datetime64[ns]
datetime     datetime64[ns]
dtype: object


Раз тип корректный, из даты формата `datetime` можно извлекать отдельные части:

In [10]:
df2["date_norm"].dt.year

0        2025
1        2025
2        2025
3        2025
4        2025
         ... 
10055    2013
10056    2013
10057    2013
10058    2013
10059    2013
Name: date_norm, Length: 10060, dtype: int64

In [11]:
df2["date_norm"].dt.month

0         4
1         4
2         4
3         4
4         3
         ..
10055    12
10056    12
10057    12
10058    12
10059    12
Name: date_norm, Length: 10060, dtype: int64

In [12]:
df2["date_norm"].dt.day

0        18
1        17
2        12
3         5
4        31
         ..
10055    24
10056    24
10057    24
10058    24
10059    24
Name: date_norm, Length: 10060, dtype: int64

> Подумайте, как могут называться атрибуты/методы для извлечения дня недели, квартала, а также часов, минут и секунд. 

In [13]:
# 0 – понедельник, 
# 6 – воскресенье

df2["date_norm"].dt.weekday

0        4
1        3
2        5
3        5
4        0
        ..
10055    1
10056    1
10057    1
10058    1
10059    1
Name: date_norm, Length: 10060, dtype: int64

In [14]:
# названия на английском

df2["date_norm"].dt.day_name()

0          Friday
1        Thursday
2        Saturday
3        Saturday
4          Monday
           ...   
10055     Tuesday
10056     Tuesday
10057     Tuesday
10058     Tuesday
10059     Tuesday
Name: date_norm, Length: 10060, dtype: object

In [15]:
# квартал 1-4

df2["date_norm"].dt.quarter

0        2
1        2
2        2
3        2
4        1
        ..
10055    4
10056    4
10057    4
10058    4
10059    4
Name: date_norm, Length: 10060, dtype: int64

In [16]:
# часы

df2["datetime"].dt.hour

0        13
1        14
2        12
3        12
4        11
         ..
10055    20
10056    20
10057    20
10058    20
10059    19
Name: datetime, Length: 10060, dtype: int64

In [17]:
# минуты

df2["datetime"].dt.minute

0        22
1        38
2        24
3        11
4        34
         ..
10055    51
10056    36
10057    35
10058    14
10059    40
Name: datetime, Length: 10060, dtype: int64

In [18]:
# секунды

df2["datetime"].dt.second

0        37
1        47
2        34
3        36
4        55
         ..
10055     4
10056    39
10057    45
10058    21
10059     4
Name: datetime, Length: 10060, dtype: int64

> Добавьте в `df2` столбец `year` с годом публикации постов и столбец `weekday` с названием дня недели.

In [19]:
df2["year"] = df2["datetime"].dt.year
df2["weekday"] = df2["datetime"].dt.day_name()
df2.head()

,timestamp,date1,date2,id,date_norm,datetime,year,weekday
0,1744982557,2025-04-18,18/04/2025,40491,2025-04-18,2025-04-18 13:22:37,2025,Friday
1,1744900727,2025-04-17,17/04/2025,40489,2025-04-17,2025-04-17 14:38:47,2025,Thursday
2,1744460674,2025-04-12,12/04/2025,40488,2025-04-12,2025-04-12 12:24:34,2025,Saturday
3,1743855096,2025-04-05,05/04/2025,40485,2025-04-05,2025-04-05 12:11:36,2025,Saturday
4,1743420895,2025-03-31,31/03/2025,40482,2025-03-31,2025-03-31 11:34:55,2025,Monday


### Часть 2: объединение датафреймов

> Объедините датафреймы `df1` и `df2` в единый датафрейм `full` по общему столбцу.

In [20]:
fin = df1.merge(df2, on = "id")
fin.head()

,id,text,nreposts,nlikes,nviews,tag,timestamp,date1,date2,date_norm,datetime,year,weekday
0,40491,"Если вы общаетесь с Богом - это хорошо,А вот е...",8,20,1003.0,ВШЭСПБ,1744982557,2025-04-18,18/04/2025,2025-04-18,2025-04-18 13:22:37,2025,Friday
1,40489,Студенты: на лекции про революцию Мэйдзи в Япо...,2,30,1888.0,Суздальцев_ВШЭ,1744900727,2025-04-17,17/04/2025,2025-04-17,2025-04-17 14:38:47,2025,Thursday
2,40488,«С точки зрения Сергея Караганова: ядерный уда...,10,24,1998.0,Суслов_ВШЭ,1744460674,2025-04-12,12/04/2025,2025-04-12,2025-04-12 12:24:34,2025,Saturday
3,40485,"Вообще, расцвет математики и матанализа пришел...",11,32,2585.0,Лебедев_ВШЭ,1743855096,2025-04-05,05/04/2025,2025-04-05,2025-04-05 12:11:36,2025,Saturday
4,40482,"Закон, как телефонный столб. Его невозможно пе...",10,40,3031.0,Гражданское_право,1743420895,2025-03-31,31/03/2025,2025-03-31,2025-03-31 11:34:55,2025,Monday


**Дополнение.** Как переделать названия дней недели – перевести их на русский? Создать словарь соответствий и подать его на вход методу `map()`:

In [21]:
print(fin["weekday"].unique())

['Friday' 'Thursday' 'Saturday' 'Monday' 'Sunday' 'Tuesday' 'Wednesday']


In [24]:
en_ru = {"Monday" : "понедельник", 
         "Tuesday" : "вторник",
         "Wednesday" : "среда", 
         "Thursday" : "четверг", 
         "Friday" : "пятница", 
         "Saturday" : "суббота",
         "Sunday" : "воскресенье"}

# применяем, заменяем и перезаписываем столбец weekday

fin["weekday"] = fin["weekday"].map(en_ru)

fin[["text", "datetime", "weekday"]].tail()

,text,datetime,weekday
10055,по маленькой кругленькой громов,2013-12-24 20:51:04,NaN
10056,Все глоки суть куздры (Данько),2013-12-24 20:36:39,NaN
10057,с какого бадуна ты это написал? (Самовол),2013-12-24 20:35:45,NaN
10058,Синдром яндекса (Шаповалов И. А),2013-12-24 20:14:21,NaN
10059,Задача тривиальна (Акимов Д.В.),2013-12-24 19:40:04,NaN
